# WorldQuant Pattern Miner — Workshop

**What you'll do in the next ~20 minutes**

1. Authenticate to the Brain API.
2. Inspect the data-field catalog the miner will sample from.
3. Walk through the single-file config that drives everything.
4. Run a short *live* mining batch (~2 minutes).
5. Query the result store with SQL (DuckDB over JSONL).
6. Iterate — change a template, re-run, compare.

**Prereqs**

- A WorldQuant Brain account (non-consultant tier is fine — this miner caps at 3 concurrent sims by default).
- The repo cloned, `.venv` activated, deps installed (`pip install -r requirements.txt && pip install duckdb`).
- `credentials/pw` containing `["your_email", "your_password"]`.

## Architecture in one diagram

```
  config.py  ──►  EXPRESSION_TEMPLATE  +  REQUIRED_TYPE / PLACEHOLDER_CATEGORY
       │
       ▼
  expressions.py  ──►  stream {a}×{b} candidates from CSV buckets
       │
       ▼
  runner.py  ──►  start ≤ MAX_CONCURRENT_SIMS, poll, fetch alpha record
       │                              ▲
       ▼                              │
  db.py  ──►  append verbatim to data/alphas.jsonl     auth.py (pause on token expiry)
       │
       ▼
  query.py  ──►  DuckDB SQL over JSONL (no schema, lossless)
```

One file == one job. Most of what we'll touch today is in `pattern_search/config.py`.

## 0. Setup — sanity check the workspace

In [ ]:
import os
import sys
from pathlib import Path

PROJECT_DIR = Path.cwd()
if PROJECT_DIR.name != "worldquant-pattern-miner":
    raise SystemExit("Open this notebook from the project root.")

sys.path.insert(0, str(PROJECT_DIR))

expected = ["pattern_search.py", "pattern_search/", "credentials/pw"]
for p in expected:
    full = PROJECT_DIR / p
    flag = "✅" if full.exists() else "❌"
    print(f"{flag}  {p}")

print("\nproject =", PROJECT_DIR)

## 1. Authenticate

`credentials/token_refresh.py` posts your `pw` to the Brain auth endpoint, handles the Persona biometric flow if needed, and writes the JWT to `credentials/brain_token.txt`. The miner reads that file on every API call.

Run the cell below. If your account requires Persona, a URL prints — open it, complete verification, the script auto-detects success.

In [ ]:
import subprocess

result = subprocess.run(
    [sys.executable, "credentials/token_refresh.py"],
    cwd=str(PROJECT_DIR),
    capture_output=True,
    text=True,
    timeout=300,
)
print(result.stdout)
print(result.stderr)

token_path = PROJECT_DIR / "credentials" / "brain_token.txt"
if token_path.exists():
    token = token_path.read_text().strip()
    print(f"Token: {token[:8]}…{token[-4:]}  ({len(token)} chars)")
else:
    print("❌ Token not written — check stderr above.")

## 2. The data-field catalog

The Brain API exposes thousands of data fields per region — price/volume, fundamentals, news/sentiment, options, analyst, etc. The miner samples *combinations* of them.

`datafields/datafields_regional_master.py` fetches the catalog and writes one CSV per type per region to `datafields/{REGION}/{VECTOR,MATRIX,GROUP}{delay}.csv`. We've pre-fetched USA delay-1 for the workshop.

In [ ]:
import pandas as pd

csv_dir = PROJECT_DIR / "datafields" / "USA"
for csv in sorted(csv_dir.glob("*.csv")):
    n = sum(1 for _ in open(csv)) - 1  # header
    print(f"{csv.name:<20} {n:>6} datafields")

print("\nFirst rows of MATRIX.csv (the bucket our default template draws from):")
matrix_csv = next(p for p in csv_dir.glob("MATRIX*.csv"))
pd.read_csv(matrix_csv).head()

## 3. The single config file

Everything the miner does is driven by one module: `pattern_search/config.py`. Three knobs matter most:

| Knob | What it does |
|---|---|
| `EXPRESSION_TEMPLATE` | The formula skeleton, e.g. `rank(ts_mean({a},10)/ts_mean({b},252))`. Each `{x}` is filled by every datafield in its bucket. |
| `SIMULATION_CONFIG` | Backtest settings (region, universe, neutralization, truncation, decay, delay). |
| `MAX_CONCURRENT_SIMS` | How many sims run in parallel. **3 is the non-consultant cap.** |

Let's print the active values.

In [ ]:
from pattern_search import config

print("EXPRESSION_TEMPLATE:")
print("  ", config.EXPRESSION_TEMPLATE)
print("\nSIMULATION_CONFIG:")
for k, v in config.SIMULATION_CONFIG.items():
    print(f"  {k:<18} = {v}")
print(f"\nMAX_CONCURRENT_SIMS = {config.MAX_CONCURRENT_SIMS}")
print(f"REQUIRED_TYPE       = {config.REQUIRED_TYPE}")
print(f"PLACEHOLDER_CATEGORY = {config.PLACEHOLDER_CATEGORY}")

**How big is the search space?** `MATRIX × MATRIX` (filtered by category 'pv') is roughly *N²* where N is the count of pv-MATRIX datafields. Even a few hundred fields produces tens of thousands of candidate expressions — far more than we'll mine in 2 minutes.

## 4. Run the miner — live

Spawn `pattern_search.py` as a subprocess for **120 seconds**, then terminate. Output streams below. With 3 concurrent sims and ~30s per sim, expect ~10–20 alphas to land in `data/alphas.jsonl`.

(In production, you just leave `python pattern_search.py` running.)

In [ ]:
import subprocess
import time

MINING_SECONDS = 120

proc = subprocess.Popen(
    [sys.executable, "-u", "pattern_search.py"],
    cwd=str(PROJECT_DIR),
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1,
)

deadline = time.time() + MINING_SECONDS
try:
    while time.time() < deadline and proc.poll() is None:
        line = proc.stdout.readline()
        if not line:
            break
        print(line, end="")
finally:
    proc.terminate()
    try:
        proc.wait(timeout=10)
    except subprocess.TimeoutExpired:
        proc.kill()
    print(f"\n[workshop] miner stopped after {MINING_SECONDS}s")

## 5. Inspect the result store with SQL

Successful alphas are appended verbatim (full Brain API payload) to `data/alphas.jsonl` — one JSON per line. **No schema, no migrations, no DB.** Query it with DuckDB directly.

In [ ]:
import duckdb

con = duckdb.connect(":memory:")
con.execute("""
  CREATE VIEW alphas AS
  SELECT * FROM read_json_auto('data/alphas.jsonl', format='newline_delimited')
""")

con.sql("SELECT count(*) AS total_alphas FROM alphas").show()

In [ ]:
# Top by Sharpe — what shape are these signals?
con.sql("""
  SELECT id, regular.code AS expression,
         "is".sharpe, "is".fitness, "is".turnover, "is".returns
  FROM alphas
  ORDER BY "is".sharpe DESC NULLS LAST
  LIMIT 10
""").show(max_width=200)

In [ ]:
# Where are alphas being rejected?
con.sql("""
  SELECT c.name, c.result, count(*) AS n
  FROM alphas, UNNEST("is".checks) AS t(c)
  GROUP BY 1, 2
  ORDER BY 1, 3 DESC
""").show()

In [ ]:
# Submission-ready: alphas with NO FAIL on any check
con.sql("""
  SELECT id, regular.code AS expression,
         "is".sharpe, "is".fitness, "is".turnover
  FROM alphas
  WHERE NOT list_contains(list_transform("is".checks, c -> c.result), 'FAIL')
  ORDER BY "is".sharpe DESC NULLS LAST
""").show(max_width=200)

In [ ]:
# Near-misses: best alphas that fail the fewest checks
con.sql("""
  SELECT id, "is".sharpe,
         len(list_filter("is".checks, c -> c.result = 'FAIL')) AS n_fail,
         list_transform(list_filter("is".checks, c -> c.result = 'FAIL'),
                        c -> c.name) AS failed_checks
  FROM alphas
  ORDER BY n_fail ASC, "is".sharpe DESC NULLS LAST
  LIMIT 10
""").show(max_width=200)

## 6. Iterate — change a template, re-run

If the check distribution above is dominated by `LOW_SHARPE` / `LOW_FITNESS`, the template is too noisy. Common fixes:

- **Smooth the signal** — wrap with `ts_decay_linear(..., 60)` or `ts_backfill(sqrt(...), 21)`.
- **Saturate outliers** — `tanh(3*(ts_rank(x,252) - 0.5))` or `zscore(...)`.
- **Sparsify** — `trade_when(ts_rank({a},252) > 0.8, ..., 0)` only trades when the factor is in the top quintile.

There are commented templates in `pattern_search/config.py` — pick one, comment-swap, rerun this notebook from cell 4. The miner skips already-simulated expressions automatically.

**Workshop exercise:** swap to this smoother variant and re-run cell 4 for another 2 minutes:

```python
EXPRESSION_TEMPLATE = "ts_backfill(sqrt(rank(ts_mean({a},5)/ts_mean({b},252))),21)"
```

Then compare the two check-distribution outputs side-by-side.

## Recap

- **Auth** — `python credentials/token_refresh.py` once; the miner reads `brain_token.txt` and pauses to re-auth on expiry.
- **Datafields** — `python datafields/datafields_regional_master.py` populates `datafields/{REGION}/`.
- **Mine** — `python pattern_search.py`. Configure via `pattern_search/config.py`.
- **Persist** — append-only JSONL at `data/alphas.jsonl`, lossless (full API payload).
- **Query** — `python query.py` (REPL) or `python query.py "SELECT …"` (one-shot DuckDB).
- **Iterate** — change template, rerun, compare.

Repo: https://github.com/JUZMEBOK/worldquant-pattern-miner